## Homework - Water Models & Water Density

Using the code below:

1. Run 100,000-step MD simulations of bulk water using the TIP3P water model and a temperature of 298 K. You can leave thermostat/barostat coupling constants unchanged from their current values.
1. Import the data in the data.csv file into MS Excel. (_NOTE: This file will get deleted & overwritten each time you run a simulation! So save them as you go._)
1. Inspect the Potential Energy, Density and Temperature data to determine whether or not your MD simulation has equilibrated for each simulation.
1. Calculate the average density, $\rho$, and its associated standard deviation, over an appropriate section of the MD trajectory.
1. Repeat steps 1-4 for temperatures between -20ºC and 50ºC, for both the TIP3P and TIP4P-Ew water models.
1. Answer the questions at the end of this notebook.

**Submit your homework as a single MS excel file. Use a separate spreadsheet for each set of simulation results, and place your answer to steps (6) and (7) on a separate spreadsheet 'summary'. Ensure that your analysis, plots and answers are appropriately labelled and presented clearly.**
    

In [ ]:
#@title Setup the Environment 🔨🔨🔨
!python -V
!pip install -q openmm MDAnalysis

# install compiler
!apt-get -qq update
!apt-get -qq install -y gfortran

# remove existing directory if it exists
import os
import shutil
if os.path.exists("packmol-master"):
    print("Removing existing packmol-master directory...")
    shutil.rmtree("packmol-master")

# download latest Packmol source
!wget -q https://github.com/m3g/packmol/archive/refs/heads/master.zip
!unzip -q master.zip

# compile
%cd packmol-master
!make
%cd ..

# add to PATH

os.environ["PATH"] = "/content/packmol-master:" + os.environ["PATH"]
!which packmol

from openmm.app import *
from openmm import *
from openmm.unit import *
from sys import stdout
import re
import time
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
#@title Choose Simulation Parameters. 🧪🧪🧪


import ipywidgets as widgets
from IPython.display import display

#SIMULATION TEMPERATURE
# create a float text widget to input variable
temp_text = widgets.FloatText(
    value=298.0,  # default value
    description='Simulation Temperature (K):',
    step=1.0,  # step size
)
# adjust the layout of the widget
temp_text.layout.width = 'auto'
temp_text.style.description_width = 'initial'
# define a function to update the variable
def update_temp_text(change):
    global simTemperature
    simTemperature = change.new
# register the update function with the widget
temp_text.observe(update_temp_text, 'value')
# display the widget
display(temp_text)
# access the selected temperature value
simTemperature = temp_text.value*kelvin

#BOXSIZE
# create a float text widget to input variable
temp_text = widgets.FloatText(
    value=26.0,  # default value
    description='Box Size (Angstrom):',
    step=1.0,  # step size
)
# adjust the layout of the widget
temp_text.layout.width = 'auto'
temp_text.style.description_width = 'initial'
# define a function to update the variable
def update_temp_text(change):
    global boxsize
    boxsize = change.new
# register the update function with the widget
temp_text.observe(update_temp_text, 'value')
# display the widget
display(temp_text)
# access the selected temperature value
boxsize = temp_text.value

#TIMESTEP
# create a float text widget to input variable
temp_text = widgets.FloatText(
    value=1.0,  # default value
    description='Time step (fs):',
    step=0.5,  # step size
)
# adjust the layout of the widget
temp_text.layout.width = 'auto'
temp_text.style.description_width = 'initial'
# define a function to update the variable
def update_temp_text(change):
    global simTimestep
    simTimestep = change.new
# register the update function with the widget
temp_text.observe(update_temp_text, 'value')
# display the widget
display(temp_text)
# access the selected temperature value
simTimestep = temp_text.value*femtoseconds

#PRESSURE
# create a float text widget to input variable
temp_text = widgets.FloatText(
    value=1.0,  # default value
    description='Simulation pressure (atm):',
    step=0.10,  # step size
)
# adjust the layout of the widget
temp_text.layout.width = 'auto'
temp_text.style.description_width = 'initial'
# define a function to update the variable
def update_temp_text(change):
    global simPressure
    simPressure = change.new
# register the update function with the widget
temp_text.observe(update_temp_text, 'value')
# display the widget
display(temp_text)
# access the selected temperature value
simPressure = temp_text.value*atmospheres

#STEPS
# create a float text widget to input variable
temp_text = widgets.FloatText(
    value=100000,  # default value
    description='# MD steps:',
    step=100,  # step size
)
# adjust the layout of the widget
temp_text.layout.width = 'auto'
temp_text.style.description_width = 'initial'
# define a function to update the variable
def update_temp_text(change):
    global simNumSteps
    simNumSteps = change.new
# register the update function with the widget
temp_text.observe(update_temp_text, 'value')
# display the widget
display(temp_text)
# access the selected temperature value
simNumSteps = temp_text.value

#Forcefield selection
ffield_dropdown = widgets.Dropdown(
    options=['TIP3P', 'TIP4P-Ew'],
    value='TIP3P', # Default value
    description='Forcefield:',
    disabled=False,
)
ffield_dropdown.layout.width = 'auto'
ffield_dropdown.style.description_width = 'initial'

def update_ffield(change):
    global ffield
    selected_value = change.new
    if selected_value == 'TIP3P':
        ffield = 'tip3p'
    elif selected_value == 'TIP4P-Ew':
        ffield = 'tip4pew'

ffield_dropdown.observe(update_ffield, 'value')
display(ffield_dropdown)

# Set initial ffield based on the default value of the dropdown
initial_ffield_value = ffield_dropdown.value
if initial_ffield_value == 'TIP3P':
    ffield = 'tip3p'
elif initial_ffield_value == 'TIP4P-Ew':
    ffield = 'tip4pew'


In [ ]:
#@title Parameter Summary

density=0.99705
water=int(boxsize**3*density*1e6*(1e-10)**3/(18.01)*6.02214e23)
print("Box size of",boxsize,"angstroms at density,",density,"g/cm3 requires ",water," water molecules")
print(f"Simulation size: {water} waters")
print(f"Box size: {boxsize} Ang.")
print(f"Simulation temperature: {simTemperature}")
print(f"Simulation pressure: {simPressure}")
print(f"Simulation time step: {simTimestep}")
print(f"# MD steps: {simNumSteps} ; Total simulation length: {simNumSteps * simTimestep} = {simNumSteps*simTimestep / 1000 / 1000 / femtoseconds * nanoseconds}.")
print(f"Water model: {ffield}")
pdbFile = 'system.pdb'

#clean up any possible previous runs
for f in pdbFile, 'system_tip4pew.pdb', 'data.dcd', 'data.csv', 'water.pdb', 'packmol.in', 'packmol.out':
  if os.path.exists(f):
    os.remove(f)



In [ ]:
#@title Build Simulation Box🔨🔨🔨

with open("packmol.in", "w") as f:
    f.write("tolerance 2.0\n")
    f.write("filetype pdb\n")
    f.write("output system.pdb\n")
    f.write("\n")
    f.write("structure water.pdb\n")
    f.write("  number {0}\n".format(int(water)))
    f.write("  inside cube 2. 2. 2. {0}\n".format(boxsize))
    f.write("end structure\n")
    f.write("\n")
    f.write("pbc {0} {0} {0}\n".format(boxsize))

text = '''ATOM      1  O   HOH A   1       4.125  13.679  13.761  1.00  0.00           O
ATOM      2  H1  HOH A   1       4.025  14.428  14.348  1.00  0.00           H
ATOM      3  H2  HOH A   1       4.670  13.062  14.249  1.00  0.00           H'''

with open('water.pdb', 'w') as f:
    f.write(text)

!packmol < packmol.in > packmol.out

In [ ]:
#@title Setup & Run MD Simulation 💻

pdb = PDBFile('system.pdb')
forcefield = ForceField(f"{ffield}.xml")


if ffield == 'tip4pew':
  modeller = Modeller(pdb.topology, pdb.positions)
  modeller.addExtraParticles(forcefield)
  PDBFile.writeFile(modeller.topology, modeller.positions, open('system_tip4pew.pdb', 'w'))
  pdb = PDBFile('system_tip4pew.pdb','M')

system = forcefield.createSystem(
    pdb.topology,
    nonbondedMethod=PME,
    nonbondedCutoff=8*angstrom,
    constraints=HBonds,
    rigidWater=True)

system.addForce(MonteCarloBarostat(simPressure, simTemperature, 1))

integrator = LangevinMiddleIntegrator(simTemperature, 1/picosecond, simTimestep)
simulation = Simulation(pdb.topology, system, integrator)
simulation.context.setPositions(pdb.positions)

simulation.reporters.append(DCDReporter('data.dcd', 10))
simulation.reporters.append(StateDataReporter(
    'data.csv',
    10,
    step=True,
    temperature=True,
    potentialEnergy=True,
    kineticEnergy=True,
    totalEnergy=True,
    volume=True,
    density=True
))

print("Running Energy Minimisation:")
t0 = time.time()
simulation.minimizeEnergy()
t1 = time.time()
minTime = t1-t0
print(f"{ffield} minimisation took {minTime} seconds")

print('Running NPT Simulation')
t0 = time.time()
simulation.step(simNumSteps)
t1 = time.time()
simTime = t1-t0
print(f"{ffield} simulation took {simTime} seconds for {simNumSteps} timesteps")


Based on the data you have generated using the calculator above, complete / answer the following:

1. Import the demperature and density data from the 'data.csv' files at each temperature into single Excel workbook.
1. Plot your calculated average TIP3P and TIP4P-Ew water densities for temperatures between -20ºC and 50 ºC with the experimental density of **supercooled** water on a single graph. Include the +/- errors provided above. _NOTE: take care in identifying whether experimental water densities refer to the solid or liquid phase at / below 0ºC._
1. What are the differences in the functional forms of the TIP3P and TIP4P-Ew models? Use diagrams and equations in your answer wherever necessary.
1. Based on your answer to (2) and (3), why does the TIP3P water model fail to predict the density of liquid water at low and high temperatures?


**Submit your homework as a single MS excel file. Use a separate spreadsheet for each set of simulation results, and place your answers on a separate spreadsheet 'summary'. Ensure that your analysis, plots and answers are appropriately labelled and presented clearly.**




_You may find the following references useful for preparing your homework:_

1. Mark, P., Nilsson, L. Structure and Dynamics of the TIP3P, SPC, and SPC/E Water Models at 298 K. _Journal of Physical Chemistry A_ **43** 9954 (2001)
1. Horn _et al._ Development of an improved four-site water model for biomolecular simulations: TIP4P-Ew. _Journal of Chemical Physics_ **120** 9665 (2004)
1. W. Wagner and A. Pruss, The IAPWS Formulation 1995 for the Thermodynamic Properties of Ordinary Water Substance for General and Scientific Use. _J. Phys. Chem. Ref. Data_ **31** 387 (2003).




